# Atividade 4 — Pandas: crescimento populacional por estado e município (2010–2022)

Tabela usada: `CD2022_Populacao_2010_Compatibilizada_20231222_1.xlsx` (IBGE, População 2010 Compatibilizada).

In [12]:
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import seaborn as sns

ModuleNotFoundError: No module named 'pandas'

## 1. Abrir a tabela

In [ ]:
path = r"H:\cotemig 2026\python\3etapa\CD2022_Populacao_2010_Compatibilizada_20231222 (1).xlsx"
df = pd.read_excel(path, engine="openpyxl", header=2)
df.columns

In [ ]:
df.head()

## 2. Descobrir as colunas de UF, município e população

O nome exato das colunas pode variar conforme a planilha, então localizamos pelo padrão do nome em vez de digitar fixo.

In [ ]:
colunas = list(df.columns)

col_cod_uf = [c for c in colunas if "cod" in str(c).lower() and "uf" in str(c).lower()][0]
col_uf = [c for c in colunas if c != col_cod_uf and ("uf" in str(c).lower() or "sigla" in str(c).lower())][0]
col_cod_mun = [c for c in colunas if "cod" in str(c).lower() and "munic" in str(c).lower()][0]
col_mun = [c for c in colunas if "munic" in str(c).lower() and "cod" not in str(c).lower()][0]

# A planilha tem duas colunas de 2010 (Sinopse e Alterações de Limites) e a de
# "Alterações de Limites" também tem "2022" no nome, então a busca precisa ser
# mais específica para não pegar a coluna errada.
col_2010_candidatos = [c for c in colunas if "2010" in str(c) and "limit" in str(c).lower()]
col_2010 = col_2010_candidatos[0] if col_2010_candidatos else [c for c in colunas if "2010" in str(c)][0]
col_2022 = [c for c in colunas if "2022" in str(c) and "2010" not in str(c)][0]

col_cod_uf, col_uf, col_cod_mun, col_mun, col_2010, col_2022

In [ ]:
df[col_2010] = pd.to_numeric(df[col_2010], errors="coerce")
df[col_2022] = pd.to_numeric(df[col_2022], errors="coerce")

## 3. População agregada por estado

In [ ]:
pop_estado = (
    df.groupby([col_cod_uf, col_uf], as_index=False)
    .agg(
        populacao_2010=(col_2010, "sum"),
        populacao_2022=(col_2022, "sum"),
    )
)
pop_estado

### Ordenar pelo crescimento entre 2010 e 2022

In [ ]:
pop_estado["crescimento"] = pop_estado["populacao_2022"] - pop_estado["populacao_2010"]
pop_estado = pop_estado.sort_values("crescimento", ascending=False).reset_index(drop=True)
pop_estado

### Salvar em CSV

In [ ]:
pop_estado.to_csv(r"H:\cotemig 2026\python\3etapa\populacao_por_estado.csv", sep=";")

## 4. Agora por município

In [ ]:
pop_municipio = (
    df.groupby([col_cod_mun, col_mun, col_uf], as_index=False)
    .agg(
        populacao_2010=(col_2010, "sum"),
        populacao_2022=(col_2022, "sum"),
    )
)
pop_municipio["crescimento"] = pop_municipio["populacao_2022"] - pop_municipio["populacao_2010"]
pop_municipio = pop_municipio.sort_values("crescimento", ascending=False).reset_index(drop=True)
pop_municipio

### Salvar em CSV

In [ ]:
pop_municipio.to_csv(r"H:\cotemig 2026\python\3etapa\populacao_por_municipio.csv", sep=";")

## 5. Gráfico: 10 estados que mais cresceram

In [ ]:
top = pop_estado.head(10).rename(columns={"populacao_2010": "2010", "populacao_2022": "2022"}).melt(
    id_vars=col_uf,
    value_vars=["2010", "2022"],
    var_name="ano",
    value_name="populacao",
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=top, x=col_uf, y="populacao", hue="ano", ax=ax)
ax.set_title("10 UFs com maior crescimento absoluto: 2010 vs 2022")
ax.set_ylabel("Habitantes")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()